In [2]:
# Instalar paquetes pandas y numpy, pip install pandas numpy
import pandas
import numpy
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


cars = pandas.read_csv('cars.csv', header=None,
                       names=['buying', 'maint', 'doors', 'persons',
                              'lug_boot', 'safety', 'acceptability'])

atributos = cars.loc[:, 'buying':'safety']  # selección de las columnas de atributos
objetivo = cars['acceptability']  # selección de la columna objetivo

# El codificador adecuado para los atributos es OrdinalEncoder, ya que permite
# trabajar con el array completo de valores de los atributos.
codificador_atributos = preprocessing.OrdinalEncoder()
codificador_atributos.fit(atributos)

# Codificación de los atributos una vez ajustado el codificador
atributos_codificados = codificador_atributos.transform(atributos)

# El codificador adecuado para la variable objetivo es LabelEncoder, que trabaja
# con la lista de sus valores, en lugar de con un array.
codificador_objetivo = preprocessing.LabelEncoder()

# El método fit_transform ajusta la codificación y la aplica a los datos a continuación
objetivo_codificado = codificador_objetivo.fit_transform(objetivo)

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = model_selection.train_test_split(
        # Conjuntos de datos a dividir, usando los mismos índices para ambos
        atributos_codificados, objetivo_codificado,
        # Valor de la semilla aleatoria, para que el muestreo sea reproducible,
        # a pesar de ser aleatorio
        random_state=12345,
        # Tamaño del conjunto de prueba
        test_size=.33,
        # Estratificamos respecto a la distribución de valores en la variable objetivo
        stratify=objetivo_codificado)

# En scikit-learn, ID3 se especifica mediante criterion='entropy'.
# Nota: La implementación nativa de sklearn soporta divisiones numéricas/ordinales.
modelo_id3 = DecisionTreeClassifier(
    criterion='entropy',  # Usa Ganancia de Información (Entropía) como en ID3
    random_state=12345
)

# Entrenamiento del modelo
modelo_id3.fit(atributos_entrenamiento, objetivo_entrenamiento)

# Evaluación del modelo
predicciones = modelo_id3.predict(atributos_prueba)

# Métricas de precisión general
precision_general = accuracy_score(objetivo_prueba, predicciones)




#########################
####    FORMULARIO   ####
#########################

# 1. Instalar paquete ipywidgets.
#    El paquete ipywidgets (también conocido como jupyter-widgets) 
#    es una biblioteca de Python que permite crear elementos gráficos interactivos

#    Instalar paquete IPython. 
#    Contiene metodos para usar etiquetas HTML.

import ipywidgets as widgets
from IPython.display import display, HTML

# crear variables que definen el ancho del formulario y sus controles
style = {'description_width': '110px'}
layout = widgets.Layout(width='320px')

# 2. Crear los elementos del formulario (selectores desplegables)
dropdowns = {
    'buying': widgets.Dropdown(
        options=[('Muy Alto', 'vhigh'), ('Alto', 'high'), ('Medio', 'med'), ('Bajo', 'low')], 
        value='high', 
        description='Precio Compra:', 
        style=style, 
        layout=layout
    ),
    'maint': widgets.Dropdown(
        options=[('Muy Alto', 'vhigh'), ('Alto', 'high'), ('Medio', 'med'), ('Bajo', 'low')], 
        value='med', 
        description='Mantenimiento:', 
        style=style, 
        layout=layout
    ),
    'doors': widgets.Dropdown(
        options=[('2 puertas', '2'), ('3 puertas', '3'), ('4 puertas', '4'), ('5 o más', '5more')], 
        value='4', 
        description='Nº de Puertas:', 
        style=style, 
        layout=layout
    ),
    'persons': widgets.Dropdown(
        options=[('2 personas', '2'), ('4 personas', '4'), ('Más de 4', 'more')], 
        value='4', 
        description='Capacidad Personas:', 
        style=style, 
        layout=layout
    ),
    'lug_boot': widgets.Dropdown(
        options=[('Pequeño', 'small'), ('Mediano', 'med'), ('Grande', 'big')], 
        value='med', 
        description='Tamaño Maletero:', 
        style=style, 
        layout=layout
    ),
    'safety': widgets.Dropdown(
        options=[('Baja', 'low'), ('Media', 'med'), ('Alta', 'high')], 
        value='high', 
        description='Nivel de Seguridad:', 
        style=style, 
        layout=layout
    )
}

# Crear los elementos del formulario (Botón del formulario para evaluar la petición)
btn_predict = widgets.Button(
    description='Evaluar Coche',
    button_style='primary',
    icon='car',
    layout=widgets.Layout(width='320px', margin='15px 0 0 0')
)

# Variable contenedora que contiene contenido de los controles del formulario
out_result = widgets.Output()

# 3. Función de enlace (Front-end con Back-end)
def evaluar_coche(b):
    with out_result:
        out_result.clear_output()
        
        # Captura los valores del formulario HTML
        nuevo_coche = pandas.DataFrame([{
            'buying': dropdowns['buying'].value,
            'maint': dropdowns['maint'].value,
            'doors': dropdowns['doors'].value,
            'persons': dropdowns['persons'].value,
            'lug_boot': dropdowns['lug_boot'].value,
            'safety': dropdowns['safety'].value
        }])
        
        # Usamos .transform() 
        nuevos_atributos_codificados = codificador_atributos.transform(nuevo_coche)

        # Realizamos la predicción numérica con el modelo ID3
        predicciones_numericas = modelo_id3.predict(nuevos_atributos_codificados)

        # Convertimos la predicción numérica de vuelta a su etiqueta original ('unacc', 'acc', etc.)
        predicciones_etiquetas = codificador_objetivo.inverse_transform(predicciones_numericas)[0]
        
        # Formato visual del resultado
        colores = {'unacc': '#e74c3c', 'acc': '#f39c12', 'good': '#3498db', 'vgood': '#2ecc71'}
        color = colores.get(predicciones_etiquetas, '#333')

        if predicciones_etiquetas == "acc":
            resultado="Aceptable"
        elif predicciones_etiquetas == "unacc":
            resultado="Inaceptable"
        elif predicciones_etiquetas == "good":
            resultado="bueno"
        else:
            resultado="Muy bueno"
        
        display(HTML(f"""
            <div style="background-color: {color}; color: white; padding: 15px; border-radius: 8px; margin-top: 15px; text-align: center; font-family: sans-serif;">
                <h3 style="margin: 0; font-size: 1.1rem;">Resultado de la Evaluación:</h3>
                <h1 style="margin: 5px 0 0 0; font-size: 2rem; text-transform: uppercase;">{resultado}</h1>
            </div>
        """))

btn_predict.on_click(evaluar_coche)

# 3. Mostrar el formulario en pantalla
formulario_box = widgets.VBox([
    widgets.HTML("<h3>🚗 Evaluación de Aceptabilidad de Vehículos</h3>"),
    *dropdowns.values(),
    btn_predict,
    out_result
])

display(formulario_box)
